# Building a Dataset of Newspapers by County Where Lynchings Occurred

In [ ]:
import pandas as pd
import ast
import os
import tarfile
import shutil
from pathlib import Path

newspapers = pd.read_csv('chron_am_newspapers.csv')
lynchings = pd.read_csv('combined_inventories.csv')
batches_df = pd.read_csv('https://raw.githubusercontent.com/MatthewKollmer/chron_am_backup/refs/heads/main/ocr_batches.csv', converters={'contents': ast.literal_eval}) # converters here ensures the lists of dictionaries in ocr_batches.csv are read as such
papers_df = pd.read_csv('https://raw.githubusercontent.com/MatthewKollmer/chron_am_backup/refs/heads/main/newspapers.csv')

This notebook completes the following steps:

1) preprocessing data of Chronicling America's digitized newspapers. This data can be downloaded here: [https://www.loc.gov/collections/chronicling-america/titles/](https://www.loc.gov/collections/chronicling-america/titles/)
2) preprocessing data from the combined version of the Tolnay-Beck-Bailey lynching inventory and the Seguin & Rigby lynching dataset. More info about these records can be found here: [https://github.com/MatthewKollmer/us_lynching_victims](https://github.com/MatthewKollmer/us_lynching_victims)
3) cross-referencing these datasets to identify newspapers within the counties where lynchings occurred.
4) retrieving any newspaper pages (OCR version) from counties where lynchings occurred. The pages must come from the four years prior to a lynching, the year of a lynching, or the two years afterwards (a rough contemporaneous window). Pages are retrieved from a local version of Chronicling America. For more info about how I built a local version of Chronicling America, visit this GitHub repo: [https://github.com/MatthewKollmer/chron_am_backup](https://github.com/MatthewKollmer/chron_am_backup)

Results:
- 483 cases with coverage
- 40.48 GB of data
- 2,519,912 OCR newspaper pages

In [ ]:
# newspapers preprocessing
state_map = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT',
    'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD',
    'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
}

newspapers['State'] = (
    newspapers['State']
    .str.strip()
    .str.title()
    .map(state_map)
)

newspapers['County'] = (
    newspapers['County']
    .str.lower()
    .str.replace(r'\b(county|parish)\b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [ ]:
newspapers

In [ ]:
lynchings['county'] = (
    lynchings['county']
    .str.lower()
    .str.replace(r'\b(county|parish)\b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

day_fixed = pd.to_numeric(lynchings['day'], errors='coerce')
day_fixed = day_fixed.replace(0, 1)

lynchings['lynch_date'] = pd.to_datetime(
    {'year': lynchings['year'], 'month': lynchings['month'], 'day': day_fixed},
    errors='coerce'
).dt.strftime('%Y-%m-%d')

In [ ]:
lynchings

In [ ]:
# matching cases to overlapping digitized newspaper coverage
matched = newspapers.merge(
    lynchings,
    left_on=['State', 'County'],
    right_on=['state', 'county'],
    how='inner',
)

matched['First Issue'] = pd.to_datetime(matched['First Issue'], errors='coerce')
matched['Last Issue'] = pd.to_datetime(matched['Last Issue'], errors='coerce')
matched['lynch_date'] = pd.to_datetime(matched['lynch_date'], errors='coerce')

matched['lynch_in_range'] = matched['lynch_date'].between(
    matched['First Issue'], matched['Last Issue'], inclusive='both'
)

matched = matched[matched['lynch_in_range']]

matched

In [ ]:
# mapping newspaper coverage to Chron Am batches directory
papers_df

In [ ]:
matched = matched.merge(
    papers_df[['LCCN', 'tarfiles']],
    on='LCCN',
    how='left'
)

matched

In [ ]:
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

matched['tarfiles'] = matched['tarfiles'].map(to_list)

matched['lynch_year'] = pd.to_datetime(matched['lynch_date'], errors='coerce').dt.year

def pick_batches(tarfiles, lynch_year):
    if not tarfiles or pd.isna(lynch_year):
        return []
    window = set(range(int(lynch_year) - 4, int(lynch_year) + 3))
    out = []
    for d in tarfiles:
        years = [y for y in d.get('years', []) if y in window]
        if years:
            out.append({'file_name': d.get('file_name'), 'years': years})
    return out

matched['batches_to_pull'] = matched.apply(
    lambda r: pick_batches(r['tarfiles'], r['lynch_year']),
    axis=1
)

In [ ]:
# removing cases with white victims to focus more intently on racially motivated lynchings
matched = matched[~matched['race'].str.contains('White', case=False, na=False)]

matched

In [ ]:
# retrieve the relevant OCR data from local version of Chron Am
TAR_ROOT = '/Volumes/t5_evo_8tb/ChronAm/tarbiz2_files'
OUT_ROOT = '/Volumes/t5_evo_8tb/ChronAm/sundown_town_data'

MAX_BYTES = 500 * 1024**3  # this is just to ensure I don't come close to overfilling my SSD (set to 500GB)
total_written = 0

class StopCopy(Exception):
    pass

# tarball -> list of (lccn, years_set, victim)
tar_to_targets = {}

for _, row in matched.iterrows():
    lccn = row['LCCN']
    victim = str(row['victim']).strip()
    batches = row.get('batches_to_pull', []) or []
    for b in batches:
        tar_name = b.get('file_name')
        years = b.get('years', [])
        if not tar_name or not years:
            continue
        tar_to_targets.setdefault(tar_name, []).append((lccn, set(map(str, years)), victim))

def member_matches(member_name, lccn, years_set):
    parts = member_name.split('/')
    if lccn not in parts:
        return False, None
    for y in years_set:
        if y in parts:
            return True, parts
    return False, None

try:
    for tar_name, targets in tar_to_targets.items():
        tar_path = os.path.join(TAR_ROOT, tar_name)
        if not os.path.exists(tar_path):
            print(f'Missing tarball: {tar_path}')
            continue

        with tarfile.open(tar_path, 'r:bz2') as tf:
            for member in tf:
                if not member.isfile() or not member.name.endswith('.txt'):
                    continue

                for lccn, years_set, victim in targets:
                    ok, parts = member_matches(member.name, lccn, years_set)
                    if not ok:
                        continue

                    lccn_idx = parts.index(lccn)
                    rel_parts = parts[lccn_idx:]
                    dest_path = os.path.join(OUT_ROOT, victim, *rel_parts)

                    if os.path.exists(dest_path):
                        break

                    # Stop if this file would push us over the 500GB cap
                    if total_written + member.size > MAX_BYTES:
                        raise StopCopy(
                            f'Stopping: {total_written/1024**3:.2f}GB written, '
                            f'next file is {member.size/1024**3:.2f}GB.'
                        )

                    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

                    f = tf.extractfile(member)
                    if f is None:
                        break
                    with f, open(dest_path, 'wb') as out_f:
                        shutil.copyfileobj(f, out_f, length=1024 * 1024)

                    total_written += member.size
                    break

except StopCopy as e:
    print(e)

In [ ]:
# results/data compiled
root = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')
count = sum(1 for p in root.iterdir() if p.is_dir())
print(f'Number of cases where we have temporally overlapping digitized pages from the same county: {count}')
# Number of cases where we have temporally overlapping digitized pages from the same county: 483

In [ ]:
dir_path = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')

if not dir_path.exists():
    raise FileNotFoundError(f'Path not found: {dir_path}')

total_bytes = sum(
    (Path(root) / name).stat().st_size
    for root, _, files in os.walk(dir_path)
    for name in files
)

print(f'Directory size: {total_bytes / (1024**3):.2f} GB')
# Directory size: 40.48 GB

In [ ]:
if not dir_path.exists():
    raise FileNotFoundError(f'Path not found: {dir_path}')

txt_count = sum(
    1
    for root, _, files in os.walk(dir_path)
    for name in files
    if name.lower().endswith('.txt')
)

print(f'Number of extracted newspaper pages: {txt_count}')
# Number of extracted newspaper pages: 2519912